# Part 4 — Feature Pipeline

**BFT65B — Big Data Analytics in Finance | Student ID: 11230945**

Construct a verified bank–decision-quarter feature table and four-quarter outcome. Financial information is selected as of the decision date; future event evidence is restricted to outcome adjudication. The notebook does not evaluate a model on the historical test.

## 1. Repository and runtime
Use a fresh CPU runtime. Java is required by the complete repository test suite.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, shutil
REPO_URL = "https://github.com/NguyenDuong0508/11230945.git"
REPO = Path("/content/11230945")
if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
else:
    assert subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=REPO, text=True).strip() == REPO_URL
    subprocess.run(["git", "pull", "--ff-only"], cwd=REPO, check=True)
print("Git revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())
java = shutil.which("java")
if java is None:
    raise RuntimeError("Java 17 or later is required")
os.environ["JAVA_HOME"] = str(Path(java).resolve().parents[1])
print(subprocess.run([java,"-version"],check=True,capture_output=True,text=True).stderr)
subprocess.run([sys.executable,"-m","pip","install","-r",str(REPO/"requirements.txt")],check=True)
def command(script,*args):
    result = subprocess.run([sys.executable,str(REPO/script),*map(str,args)],cwd=REPO,text=True,capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()

## 2. Mandatory offline validation
Run all repository tests, including Parquet and Spark integration. Do not proceed after a failure or skipped test.

In [ ]:
import xml.etree.ElementTree as ET
TESTS_PASSED = False
result_path = Path("/content/part4_tests.xml")
result_path.unlink(missing_ok=True)
result = subprocess.run([sys.executable,"-m","pytest","-q","-ra","--tb=short","--show-capture=no",f"--junitxml={result_path}","tests"],cwd=REPO,env=dict(os.environ,PYTEST_DISABLE_PLUGIN_AUTOLOAD="1",SPARK_LOCAL_IP="127.0.0.1",PYSPARK_PYTHON=sys.executable),capture_output=True,text=True)
print(result.stdout)
if result.stderr: print(result.stderr)
if not result_path.is_file(): raise RuntimeError("Missing pytest report")
cases=list(ET.parse(result_path).getroot().iter("testcase"))
problems=[(case,node) for case in cases for node in list(case) if node.tag in ("failure","error","skipped")]
for case,node in problems:
    print(case.get("classname"),case.get("name"),node.tag,node.get("message"),node.text or "")
if result.returncode or not cases or problems: raise RuntimeError("Offline validation failed; inspect diagnostics")
TESTS_PASSED=True
print(f"Mandatory offline validation: PASS ({len(cases)} tests, zero skips)")

## 3. Verified upstream input
The Gold layer uses the original Silver panel and dated event ledger. Processing benchmarks inform the engine choice but are not feature data.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
WORKSPACE=Path('/content/drive/MyDrive/11230945')
assert TESTS_PASSED
assert not (WORKSPACE/'pending_source.json').exists()
command('run_part2.py','verify','--workspace',WORKSPACE)
pointer=json.loads((WORKSPACE/'active_panel.json').read_text())
SILVER=WORKSPACE/pointer['path']
silver_manifest=json.loads((SILVER/'manifest.json').read_text())
assert silver_manifest['state']=='VERIFIED'
print('Silver manifest:',pointer['manifest_sha256'])
print('Verified source rows:',f"{silver_manifest['summary']['panel_rows']:,}")
print('Financial report quarters:',silver_manifest['summary']['quarters'])
processing_pointer=WORKSPACE/'active_processing.json'
if processing_pointer.exists():
    pp=json.loads(processing_pointer.read_text())
    pm=json.loads((WORKSPACE/pp['path']/'manifest.json').read_text())
    assert pm['silver_manifest_sha256']==pointer['manifest_sha256']
    print('Part 3 lineage agrees with Silver; benchmark data are not used.')
else:
    print('Part 3 evidence pointer not present; no benchmark data are required by Gold.')

## 4. Fixed information and split protocol
Decision dates use the latest report available under the 90-day assumption, with a maximum two-quarter age. Exact historical lags are relative to the selected reporting date. The four-quarter label starts at the decision date. The split is fixed before construction; the 2008–2014 historical test is not a prospectively blind external sample.

In [ ]:
sys.path.insert(0,str(REPO/'src'))
from fdic_ews.features.policy import *
from fdic_ews.features.financial import FEATURES,DETAILS
from fdic_ews.features.pipeline import COLUMNS
import pandas as pd
import numpy as np
from IPython.display import display
assert_split()
protocol=pd.DataFrame([
    ('Training','1984Q1–2000Q4'),('Purge','2001Q1–2001Q4'),
    ('Validation','2002Q1–2006Q4'),('Purge','2007Q1–2007Q4'),
    ('Historical test','2008Q1–2014Q4'),('Unused post-test','2015Q1–2025Q2')],columns=['Role','Decision quarters'])
display(protocol)
print('Raw predictor count:',len(FEATURES))
print('Gold columns:',len(COLUMNS))
print('Availability lag:',LAG,'days; maximum report age:',MAX_AGE,'quarters')
print('Last decision:',LAST,'; latest outcome:',STUDY_END)

## 5. Build and independently verify Gold
The build publishes an active pointer only after all evidence and the 50-failure gate pass. It does not modify upstream data. The verify command recomputes feature and outcome tables from Silver. A rerun creates a separate Gold run.

In [ ]:
def stream_command(script,*args):
    proc=subprocess.Popen([sys.executable,str(REPO/script),*map(str,args)],cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in proc.stdout:
        print(line,end='',flush=True)
    if proc.wait()!=0: raise RuntimeError(f'{script} failed; inspect the preceding output')
stream_command('run_part4.py','build','--workspace',WORKSPACE)
stream_command('run_part4.py','verify','--workspace',WORKSPACE)
feature_pointer=json.loads((WORKSPACE/'active_features.json').read_text())
GOLD=WORKSPACE/feature_pointer['path']
gold_manifest=json.loads((GOLD/'manifest.json').read_text())
assert gold_manifest['state']=='VERIFIED'
assert gold_manifest['silver_manifest_sha256']==pointer['manifest_sha256']
print('Gold manifest:',feature_pointer['manifest_sha256'])
print(json.dumps(gold_manifest['summary'],indent=2))

## 6. Risk-set, label and split accounting
Positive bank–quarters can refer to the same failure. Distinguish them from distinct CERT/date events and distinct institutions. The holdout is inspected only for accounting, class presence and the required failure count. Censored and unused rows remain in Gold.

In [ ]:
audit=pd.read_csv(GOLD/'quarter_audit.csv')
splits=pd.read_csv(GOLD/'split_summary.csv')
reasons=pd.read_csv(GOLD/'label_reasons.csv')
order=['train','purge_train_validation','validation','purge_development_test','test','unused_post_test']
splits=splits.set_index('split').loc[order].reset_index()
assert (audit.candidate_rows==audit.risk_rows+audit.prior_exit+audit.prior_exit_conflict+audit.report_unavailable_or_stale).all()
assert (audit.risk_rows==audit.labelled+audit.censored).all()
assert (audit.labelled==audit.positive+audit.negative).all()
summary=gold_manifest['summary']
assert summary['test_distinct_failure_certs_in_calendar_period']>=MIN_TEST_FAILURES
print('Table 1. Decision-quarter population and outcome accounting')
display(splits[['split','candidate_rows','risk_rows','labelled','positive','negative','censored','distinct_positive_events','distinct_positive_certs']])
print('Table 2. Test-calendar failure-count gate')
display(pd.DataFrame([{'linked_events':summary['test_linked_failure_events_in_calendar_period'],'represented_events':summary['test_distinct_failure_events_in_calendar_period'],'represented_failure_CERTs':summary['test_distinct_failure_certs_in_calendar_period'],'unrepresented_linked_events':summary['test_unrepresented_linked_failure_events_in_calendar_period']}]))
print('Table 3. Development outcome reasons')
dev_reasons=reasons[reasons.split.isin(['train','validation'])].pivot_table(index='reason',columns='split',values='rows',aggfunc='sum',fill_value=0)
display(dev_reasons)

## 7. Temporal and financial feature diagnostics
Feature formulas are fixed independently of test results. Training diagnostics include missingness, constants and the capital-zero treatment. Conditional YTD flow candidates are not predictors.

In [ ]:
dictionary=pd.read_csv(GOLD/'feature_dictionary.csv')
quality=pd.read_csv(GOLD/'training_feature_quality.csv')
assert dictionary.feature.tolist()==list(FEATURES)
print('Table 4. Economic feature dictionary')
display(dictionary)
print('Table 5. Training feature quality')
display(quality.sort_values('missing_pct',ascending=False).round({'missing_pct':3}))
import matplotlib.pyplot as plt
train_missing=quality.sort_values('missing_pct',ascending=True)
fig,ax=plt.subplots(figsize=(9,7))
ax.barh(train_missing.feature,train_missing.missing_pct,color='#365c72')
ax.set(xlabel='Missing training observations (%)',title='Training feature missingness')
ax.grid(axis='x',alpha=.2); fig.tight_layout(); plt.show(); plt.close(fig)
# Development-only evolution; holdout is not used for feature diagnostics.
dev=audit[audit.split.isin(['train','validation'])].copy(); dev['date']=pd.to_datetime(dev.decision_date)
fig,ax=plt.subplots(figsize=(9,3))
ax.plot(dev.date,100*dev.positive/dev.labelled.replace(0,np.nan),color='#365c72')
ax.set(xlabel='Decision quarter',ylabel='Positive labels (%)',title='Development label prevalence'); ax.grid(alpha=.2)
fig.tight_layout(); plt.show(); plt.close(fig)

## 8. Training-only preprocessing and sensitivity
The fitted pipelines are demonstrations of the initial training-only fit. No holdout feature distributions, holdout transformations or model scores are used to choose a treatment. Development sensitivity quantifies candidate coverage under a 120-day lag and additional negatives under the unconfirmed-exit assumption.

In [ ]:
prep=json.loads((GOLD/'preprocessing.json').read_text())
print('Table 6. Training-only preprocessing')
rows=[]
for name,info in prep.items():
    assert info['training_max']<=str(TRAIN_END)
    rows.append({'specification':name,'training_rows':info['training_rows'],'first_training_decision':info['training_min'],'last_training_decision':info['training_max'],'raw_features':len(info['feature_names']),'transformed_features':len(info['transformed_names']),'missing_indicators':len(info['missing_indicators']),'all_null_features':', '.join(info['all_null_features']) or 'None','constant_or_empty':', '.join(info['constant_or_empty_features']) or 'None'})
display(pd.DataFrame(rows))
print('Table 7. Capital-zero sensitivity (training)')
from fdic_ews.features.financial import CAPITAL
cap=quality[quality.feature.isin(CAPITAL)].copy()
cap['missing_if_masked_pct']=100*cap.missing_if_capital_zeros_masked/cap.train_rows
display(cap[['feature','train_rows','missing_pct','missing_if_masked_pct']].round(3))
print('Table 8. Development-only coverage and candidate-exit sensitivity')
display(dev.groupby('split',sort=False)[['candidate_rows','lag120_candidate_rows','candidate_exit_zero_extra_negatives']].sum().astype('int64').reset_index())
print('No treatment is chosen using test performance.')

## 9. Source-linked temporal case and handoff
CERT 7213 illustrates the distinction between a decision date and the source report. The Part 2 audit identifies assistance outside the 2006Q4 four-quarter outcome horizon. Inspect only this development case; no holdout scores are produced.

In [ ]:
import pyarrow.parquet as pq
case_date='2006-12-31'
e=gold_manifest['panel'][case_date]
case=pq.ParquetFile(GOLD/e['path']).read(columns=['CERT','DECISION_DATE','REPORT_DATE','AVAILABLE_AT','HORIZON_END','Y_FAIL_4Q','LABEL_REASON']).to_pandas()
case=case.loc[case.CERT.eq(7213)]
if not case.empty:
    display(case)
    assert pd.to_datetime(case.AVAILABLE_AT).le(pd.to_datetime(case.DECISION_DATE)).all()
    assert pd.to_datetime(case.HORIZON_END).eq(pd.Timestamp('2007-12-31')).all()
else:
    print('CERT 7213 not in this decision risk set; inspect the verified source and risk accounting.')
print('Gold state:',gold_manifest['state'])
print('Decision quarters:',summary['decision_quarters'])
print('Risk rows:',f"{summary['risk_rows']:,}")
print('Labelled rows:',f"{summary['labelled_rows']:,}")
print('Model evaluation: NOT PERFORMED')
print('Part 5 input pointer:',WORKSPACE/'active_features.json')

## 10. Handoff
Part 5 reads the verified raw Gold table and uses the ordered predictor allowlist. Fit a fresh preprocessing pipeline inside each chronological fold, lock the feature/model/threshold choices on development, and evaluate the fixed historical test once. Candidate-exit, lag and capital-zero sensitivity are not selected using test scores. Save this executed notebook to the repository with its outputs. Keep all data and serialized estimators in the workspace.

See `docs/part4_protocol.md` for the complete methodology, assumptions, limitations and references.